In [9]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

working_dir = Path.cwd()
while working_dir.name != 'CausalMTR-BC':
    working_dir = working_dir.parent
    if working_dir == Path.home():
        raise FileNotFoundError("Base directory 'CausalMTR-BC' not found")
os.chdir(working_dir)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from cmtr_bc.waymo_iterator import ProcessedTrajectoryIterator

In [11]:
it_10_shuffle = ProcessedTrajectoryIterator("/scratch/cluster/abitpal/CausalMTR-BC/data/1s_1s_mini_train", 500_000, prefetch_length=10, shuffle=True)
it_10 = ProcessedTrajectoryIterator("/scratch/cluster/abitpal/CausalMTR-BC/data/1s_1s_mini_train", 500_000, prefetch_length=10, shuffle=False)

In [12]:
%timeit -n 1 -r 1 next(iter(it_10_shuffle))

7.26 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [8]:
%timeit -n 1 -r 1 next(iter(it_10))

5.33 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [84]:
%timeit next(iter(it_5))

KeyboardInterrupt: 

In [8]:
import torch

In [10]:
%timeit torch.load("/scratch/cluster/abitpal/CausalMTR-BC/data/1s_1s_mini_train/saved_samples_0500.pt", weights_only=False)

976 ms ± 56.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
data = torch.load("/scratch/cluster/abitpal/CausalMTR-BC/data/1s_1s_mini_train/saved_samples_0500.pt", weights_only=False)

In [12]:
import random

In [13]:
%timeit random.shuffle(data)

92.4 μs ± 765 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [21]:
a = []
def stuff():
    global a
    a += data

In [22]:
%timeit stuff()

2.73 μs ± 1.81 μs per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [23]:
import threading

In [53]:
root = Path.cwd() / "data/1s_1s_mini_train"
files = [os.path.join(root, f) for f in os.listdir(root) if os.path.isfile(os.path.join(root, f))]

In [74]:
import torch.multiprocessing as mp

def loadstuff(file): 
    torch.load(file, weights_only=False)

def dostuffthreaded(n): 
    with mp.Pool(processes=n) as p: 
        p.map(loadstuff, files[:n])

def dostuffnormal(n): 
    for i in range(n): 
        loadstuff(files[i])

In [76]:
%timeit -n 1 -r 1 dostuffthreaded(5)

2.06 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [77]:
%timeit -n 1 -r 1 dostuffnormal(5)

2.98 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [86]:
len(files)

1000